<a href="https://colab.research.google.com/github/semhfe/4DGaussians-Enhanced/blob/master/notebooks/4DGS_Enhanced.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎬 4DGaussians-Enhanced: Google Colab Workflow

Bu notebook, 4D Gaussian Splatting modellerini Google Colab'de eğitmek için eksiksiz bir iş akışı sağlar.

## Özellikler
- ✅ SAM2.1 + YOLO ile otomatik maske oluşturma
- ✅ COLMAP desteği (ham resimlerden kamera poz hesaplama)
- ✅ Maske önizleme ve doğrulama
- ✅ Maske-ağırlıklı loss ile eğitim
- ✅ Eğitim ön ayarları (quick_test, standard, high_quality, fast_motion)
- ✅ Video render ve PLY export

## Donanım Gereksinimleri
- Önerilen: A100 (Colab Pro)
- Minimum: T4 (Ücretsiz) - quick_test preset kullanın

---

## 📦 Cell 1: Kurulum (Installation)

Tüm bağımlılıkları kurar: 4DGaussians, SAM2, COLMAP, C++ submodule'ler

In [ ]:
import os
import sys

print("="*60)
print("🚀 4DGaussians-Enhanced Kurulum")
print("="*60)

# Step 1: Clone repository
if not os.path.exists('/content/4DGaussians-Enhanced'):
    print("\n📥 Repoyu klonluyorum...")
    !git clone https://github.com/semhfe/4DGaussians-Enhanced.git /content/4DGaussians-Enhanced
    %cd /content/4DGaussians-Enhanced
else:
    print("\n✅ Repo zaten mevcut")
    %cd /content/4DGaussians-Enhanced

# Step 2: Install COLMAP
print("\n📦 COLMAP kuruluyor...")
!apt-get update -qq
!apt-get install -qq -y colmap
print("✅ COLMAP kuruldu")

# Step 3: Install Python dependencies
print("\n📦 Python bağımlılıkları kuruluyor...")
!pip install -q -r requirements.txt
print("✅ Python bağımlılıkları kuruldu")

# Step 4: Run colab setup script (Ninja + SAM2 from GitHub)
print("\n🔧 Colab setup çalıştırılıyor...")
!python scripts/colab_setup.py

# Step 5: Initialize and patch submodules
print("\n📦 Submodule'ler kuruluyor...")
!git submodule update --init --recursive

# Re-run patching after submodules are initialized
print("\n🔧 C++ dosyaları patch'leniyor...")
!python -c "from scripts.colab_setup import patch_cpp_files; patch_cpp_files()"

# Install submodules
print("\n📦 Gaussian rasterization kuruluyor...")
!pip install -q ./submodules/depth-diff-gaussian-rasterization

print("\n📦 Simple-knn kuruluyor...")
!pip install -q ./submodules/simple-knn

# Step 6: Download SAM2.1 configs and models
print("\n📥 SAM2.1 config ve model indiriliyor...")
!python scripts/download_sam2.py --model-size large --checkpoint-dir /content/checkpoints

print("\n" + "="*60)
print("✅ Kurulum tamamlandı!")
print("="*60)
print("\n📝 Sonraki adım: Cell 2'yi çalıştırarak verilerinizi hazırlayın")

## 📁 Cell 2: Veri Hazırlama (Data Setup)

Google Drive'ı mount eder, veriyi unzip eder ve formatı doğrular.

In [ ]:
from google.colab import drive
import os
import zipfile
import shutil

print("="*60)
print("📁 Veri Hazırlama")
print("="*60)

# Step 1: Mount Google Drive
print("\n📂 Google Drive mount ediliyor...")
drive.mount('/content/drive')
print("✅ Drive mount edildi")

# Step 2: Configure paths
# BURADAN DÜZENLEYIN: Veri yollarınızı belirtin
DATA_SOURCE = "/content/drive/MyDrive/4dgs_data/my_scene.zip"  # Zip dosyası veya klasör yolu
OUTPUT_BASE = "/content/drive/MyDrive/4dgs_outputs"  # Çıktıların kaydedileceği Drive klasörü

# Local processing paths (faster than Drive)
LOCAL_DATA = "/content/data/my_scene"  # Lokal veri klasörü (işleme için)
LOCAL_OUTPUT = "/content/output"  # Lokal çıktı (eğitim için)

# Step 3: Extract or copy data to local disk
os.makedirs(LOCAL_DATA, exist_ok=True)

if DATA_SOURCE.endswith('.zip'):
    if not os.path.exists(DATA_SOURCE):
        print(f"\n❌ Hata: Zip dosyası bulunamadı: {DATA_SOURCE}")
        print("   Lütfen DATA_SOURCE değişkenini güncelleyin")
    else:
        print(f"\n📦 Zip açılıyor: {DATA_SOURCE}")
        with zipfile.ZipFile(DATA_SOURCE, 'r') as zip_ref:
            zip_ref.extractall(LOCAL_DATA)
        print(f"✅ Zip açıldı: {LOCAL_DATA}")
else:
    if not os.path.exists(DATA_SOURCE):
        print(f"\n❌ Hata: Klasör bulunamadı: {DATA_SOURCE}")
        print("   Lütfen DATA_SOURCE değişkenini güncelleyin")
    else:
        print(f"\n📂 Veri kopyalanıyor: {DATA_SOURCE} -> {LOCAL_DATA}")
        if os.path.exists(LOCAL_DATA):
            shutil.rmtree(LOCAL_DATA)
        shutil.copytree(DATA_SOURCE, LOCAL_DATA)
        print(f"✅ Veri kopyalandı")

# Step 4: Detect data format
print("\n🔍 Veri formatı algılanıyor...")
contents = os.listdir(LOCAL_DATA)
print(f"   İçerik: {contents}")

data_format = None
if 'transforms_train.json' in contents:
    data_format = 'blender'
    print("✅ Format: Blender/NeRF Synthetic")
elif 'sparse' in contents or 'images' in contents:
    data_format = 'colmap'
    print("✅ Format: COLMAP")
elif any('cam' in item for item in contents):
    data_format = 'multicam'
    print("✅ Format: Multi-camera (cam01, cam02, ...)")
elif len([f for f in contents if f.endswith(('.jpg', '.png'))]) > 0:
    data_format = 'raw_images'
    print("✅ Format: Ham resimler (COLMAP gerekli)")
else:
    print("⚠️  Format belirlenemedi. Klasör yapısını kontrol edin.")

# Step 5: Create output directory
os.makedirs(LOCAL_OUTPUT, exist_ok=True)
os.makedirs(OUTPUT_BASE, exist_ok=True)

print("\n" + "="*60)
print("✅ Veri hazırlama tamamlandı!")
print("="*60)
print(f"\n📁 Lokal veri: {LOCAL_DATA}")
print(f"📁 Lokal çıktı: {LOCAL_OUTPUT}")
print(f"📁 Drive çıktı: {OUTPUT_BASE}")
print(f"\n📊 Format: {data_format}")

if data_format == 'raw_images':
    print("\n⚠️  Ham resimler tespit edildi!")
    print("   Cell 3'ü çalıştırarak COLMAP ile kamera pozlarını hesaplayın")
else:
    print("\n📝 Sonraki adım: Cell 4'ü çalıştırarak maske oluşturun")

## 🎯 Cell 3: COLMAP İşleme (Opsiyonel)

**Sadece ham resimleriniz varsa çalıştırın!**

COLMAP ile kamera pozlarını ve sparse point cloud'u hesaplar.

In [ ]:
# Bu cell'i sadece data_format == 'raw_images' ise çalıştırın

RUN_COLMAP = False  # @param {type:"boolean"}

if RUN_COLMAP:
    print("="*60)
    print("🎯 COLMAP İşleme")
    print("="*60)
    
    # Run COLMAP pipeline
    !python scripts/run_colmap.py \
        --source_path {LOCAL_DATA} \
        --output_path {LOCAL_DATA}/colmap_output \
        --images_dir images
    
    print("\n✅ COLMAP işleme tamamlandı!")
    print("📝 Sonraki adım: Cell 4'ü çalıştırarak maske oluşturun")
else:
    print("⏭️  COLMAP atlandı (RUN_COLMAP=False)")
    print("   Ham resimleriniz varsa RUN_COLMAP=True yapın ve tekrar çalıştırın")

## 🎭 Cell 4: SAM2 Maske Oluşturma

YOLO + SAM2.1 ile otomatik maske oluşturur.

In [ ]:
import sys
sys.path.insert(0, '/content/4DGaussians-Enhanced')

from utils.sam2_utils import generate_masks_for_scene

print("="*60)
print("🎭 SAM2 Maske Oluşturma")
print("="*60)

# SAM2 Konfigürasyonu
# Tespit etmek istediğiniz nesneler (virgülle ayrılmış)
# Örnekler:
#   "person,human" - insan
#   "insan,kişi" - Türkçe (otomatik İngilizce'ye çevrilir)
#   "person,human,bag,backpack" - insan ve aksesuarlar
#   "car,vehicle" - araç
#   "dog,pet" - evcil hayvan
DETECTION_PROMPT = "person,human"  # @param {type:"string"}

# Model boyutu - büyük = daha iyi kalite ama yavaş
MODEL_SIZE = "large"  # @param ["tiny", "small", "base", "large"]

# Güven eşiği - yüksek = daha katı tespit
CONFIDENCE_THRESHOLD = 0.5  # @param {type:"slider", min:0.1, max:0.9, step:0.05}

# Her N frame'de bir işle - 1 = tüm frame'ler, daha yüksek = daha hızlı
EVERY_N_FRAMES = 1  # @param {type:"integer"}

print(f"\n🎯 Parametreler:")
print(f"   Prompt: {DETECTION_PROMPT}")
print(f"   Model: {MODEL_SIZE}")
print(f"   Eşik: {CONFIDENCE_THRESHOLD}")
print(f"   Her N frame: {EVERY_N_FRAMES}")

# Maske oluştur
stats = generate_masks_for_scene(
    source_path=LOCAL_DATA,
    mask_folder="masks",
    prompt=DETECTION_PROMPT,
    threshold=CONFIDENCE_THRESHOLD,
    every_n=EVERY_N_FRAMES,
    model_size=MODEL_SIZE,
    device="cuda",
    checkpoint_dir="/content/checkpoints"
)

print("\n" + "="*60)
print("✅ Maske oluşturma tamamlandı!")
print("="*60)
print(f"\n📊 İstatistikler: {stats}")
print("\n📝 Sonraki adım: Cell 5 ile maskeleri önizleyin")

## 👀 Cell 5: Maske Önizleme

Oluşturulan maskeleri kontrol edin.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
import glob
from ipywidgets import interact, IntSlider

print("="*60)
print("👀 Maske Önizleme")
print("="*60)

# Kamera klasörlerini bul
cam_folders = sorted(glob.glob(os.path.join(LOCAL_DATA, "cam*")))
if not cam_folders:
    # Tek klasör yapısı (images/)
    images_folder = os.path.join(LOCAL_DATA, "images")
    if os.path.exists(images_folder):
        cam_folders = [images_folder]

if not cam_folders:
    print("❌ Kamera klasörü bulunamadı")
else:
    def preview_mask(camera_idx=0, frame_idx=0):
        cam_folder = cam_folders[camera_idx]
        cam_name = os.path.basename(cam_folder)
        
        # Frame dosyalarını bul
        frame_files = sorted(glob.glob(os.path.join(cam_folder, "frame_*.jpg")))
        frame_files.extend(sorted(glob.glob(os.path.join(cam_folder, "frame_*.png"))))
        frame_files.extend(sorted(glob.glob(os.path.join(cam_folder, "*.jpg"))))
        frame_files.extend(sorted(glob.glob(os.path.join(cam_folder, "*.png"))))
        frame_files = sorted(list(set(frame_files)))
        
        if frame_idx >= len(frame_files):
            print(f"Frame {frame_idx} bulunamadı (toplam {len(frame_files)} frame)")
            return
        
        frame_path = frame_files[frame_idx]
        frame_name = os.path.basename(frame_path)
        
        # Maske yolunu bul
        if "frame_" in frame_name:
            mask_name = frame_name.replace("frame_", "mask_").replace(".jpg", ".png")
        else:
            mask_name = f"mask_{frame_name}".replace(".jpg", ".png")
        
        mask_path = os.path.join(cam_folder, "masks", mask_name)
        
        # Resmi yükle
        image = np.array(Image.open(frame_path).convert('RGB'))
        
        # Maskeyi yükle
        if os.path.exists(mask_path):
            mask = np.array(Image.open(mask_path).convert('L'))
            # Overlay oluştur
            overlay = image.copy()
            # Foreground'u yeşile boya
            overlay[:,:,1] = np.where(mask > 128, np.minimum(overlay[:,:,1] + 50, 255), overlay[:,:,1])
        else:
            mask = np.zeros(image.shape[:2], dtype=np.uint8)
            overlay = image
            print(f"⚠️  Maske bulunamadı: {mask_path}")
        
        # Görselleştir
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        
        axes[0].imshow(image)
        axes[0].set_title(f"{cam_name} - Frame {frame_idx}\nOrijinal")
        axes[0].axis('off')
        
        axes[1].imshow(mask, cmap='gray')
        axes[1].set_title("Maske\n(Beyaz=Ön plan, Siyah=Arka plan)")
        axes[1].axis('off')
        
        axes[2].imshow(overlay)
        axes[2].set_title("Overlay\n(Yeşil = Ön plan)")
        axes[2].axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # İstatistikler
        fg_pixels = np.sum(mask > 128)
        total_pixels = mask.size
        fg_percent = 100 * fg_pixels / total_pixels
        print(f"📊 Ön plan kapsamı: {fg_percent:.1f}%")
    
    # Frame sayısını bul
    first_cam_frames = glob.glob(os.path.join(cam_folders[0], "*.jpg"))
    first_cam_frames.extend(glob.glob(os.path.join(cam_folders[0], "*.png")))
    num_frames = len(first_cam_frames)
    
    # Interaktif önizleme
    print(f"\n📸 {len(cam_folders)} kamera, {num_frames} frame bulundu\n")
    interact(
        preview_mask,
        camera_idx=IntSlider(min=0, max=len(cam_folders)-1, step=1, value=0, description='Kamera:'),
        frame_idx=IntSlider(min=0, max=num_frames-1, step=1, value=0, description='Frame:')
    )

print("\n📝 Maskeler iyi görünüyorsa Cell 6'ya geçin")
print("   Maskeler kötüyse Cell 4'e dönüp parametreleri ayarlayın")

## ⚙️ Cell 6: Eğitim Konfigürasyonu

Eğitim parametrelerini ayarlayın.

In [ ]:
print("="*60)
print("⚙️  Eğitim Konfigürasyonu")
print("="*60)

# Eğitim Presetleri
PRESETS = {
    "quick_test": {
        "iterations": 14000,
        "coarse_iterations": 2000,
        "w_fg": 1.0,
        "w_bg": 0.1,
        "description": "Hızlı test (~30 dk A100, ~1 saat T4)"
    },
    "standard": {
        "iterations": 30000,
        "coarse_iterations": 3000,
        "w_fg": 1.0,
        "w_bg": 0.1,
        "description": "Dengeli kalite/hız (~1.5 saat A100)"
    },
    "high_quality": {
        "iterations": 60000,
        "coarse_iterations": 5000,
        "w_fg": 1.2,
        "w_bg": 0.05,
        "net_width": 128,
        "description": "En iyi kalite (~3-4 saat A100)"
    },
    "fast_motion": {
        "iterations": 45000,
        "coarse_iterations": 3000,
        "w_fg": 1.5,
        "w_bg": 0.1,
        "defor_depth": 2,
        "time_smoothness_weight": 0.005,
        "description": "Dans/aksiyon sahneleri için (~2 saat A100)"
    }
}

# Preset seçin
PRESET = "standard"  # @param ["quick_test", "standard", "high_quality", "fast_motion"]

# Temel parametreler
USE_MASK_LOSS = True  # @param {type:"boolean"}
ITERATIONS = PRESETS[PRESET]["iterations"]  # @param {type:"integer"}
W_FG = PRESETS[PRESET]["w_fg"]  # @param {type:"number"}
W_BG = PRESETS[PRESET]["w_bg"]  # @param {type:"number"}

# Gelişmiş parametreler
BATCH_SIZE = 1
LAMBDA_DSSIM = 0.0
DENSIFY_UNTIL_ITER = 15000
NET_WIDTH = PRESETS[PRESET].get("net_width", 64)
DEFOR_DEPTH = PRESETS[PRESET].get("defor_depth", 1)
TIME_SMOOTHNESS_WEIGHT = PRESETS[PRESET].get("time_smoothness_weight", 0.01)
COARSE_ITERATIONS = PRESETS[PRESET]["coarse_iterations"]

print(f"\n🎯 Seçilen preset: {PRESET}")
print(f"   {PRESETS[PRESET]['description']}")
print("\n📊 Konfigürasyon:")
print(f"   Iterasyon sayısı: {ITERATIONS}")
print(f"   Maske-ağırlıklı loss: {USE_MASK_LOSS}")
if USE_MASK_LOSS:
    print(f"   Ön plan ağırlığı (w_fg): {W_FG}")
    print(f"   Arka plan ağırlığı (w_bg): {W_BG}")

print("\n" + "="*60)
print("✅ Konfigürasyon hazır!")
print("="*60)
print("\n📝 Sonraki adım: Cell 7 ile eğitimi başlatın")

## 🚀 Cell 7: Eğitim

Model eğitimini başlatır. Eğitim lokal diskte yapılır, sonunda Drive'a kopyalanır.

In [ ]:
import time
import shutil

print("="*60)
print("🚀 Eğitim Başlıyor")
print("="*60)

# Komut oluştur
cmd_parts = [
    "python /content/4DGaussians-Enhanced/train.py",
    f"--source_path {LOCAL_DATA}",
    f"--model_path {LOCAL_OUTPUT}",
    f"--iterations {ITERATIONS}",
    f"--coarse_iterations {COARSE_ITERATIONS}",
    f"--batch_size {BATCH_SIZE}",
    f"--lambda_dssim {LAMBDA_DSSIM}",
    f"--densify_until_iter {DENSIFY_UNTIL_ITER}",
    f"--net_width {NET_WIDTH}",
    f"--defor_depth {DEFOR_DEPTH}",
    f"--time_smoothness_weight {TIME_SMOOTHNESS_WEIGHT}",
]

if USE_MASK_LOSS:
    cmd_parts.append("--use_mask_loss")
    cmd_parts.append(f"--w_fg {W_FG}")
    cmd_parts.append(f"--w_bg {W_BG}")

cmd = " ".join(cmd_parts)

print(f"\n📝 Komut:")
print(cmd)
print()

start_time = time.time()

# Eğitimi başlat
!{cmd}

elapsed = time.time() - start_time

print("\n" + "="*60)
print("✅ Eğitim tamamlandı!")
print("="*60)
print(f"⏱️  Süre: {elapsed/3600:.2f} saat")
print(f"📁 Lokal model: {LOCAL_OUTPUT}")

# Drive'a kopyala
print("\n📤 Model Drive'a kopyalanıyor...")
scene_name = os.path.basename(LOCAL_DATA)
drive_output = os.path.join(OUTPUT_BASE, scene_name)

if os.path.exists(drive_output):
    print(f"   Eski çıktı siliniyor: {drive_output}")
    shutil.rmtree(drive_output)

print(f"   Kopyalanıyor: {LOCAL_OUTPUT} -> {drive_output}")
shutil.copytree(LOCAL_OUTPUT, drive_output)
print(f"✅ Model Drive'a kopyalandı: {drive_output}")

print("\n📝 Sonraki adım: Cell 8 ile render yapın")

## 🎥 Cell 8: Render

Eğitilmiş modelden video render eder.

In [ ]:
import glob
from IPython.display import Video, display

print("="*60)
print("🎥 Video Render")
print("="*60)

# Render komutu
render_cmd = f"""python /content/4DGaussians-Enhanced/render.py \
    --source_path {LOCAL_DATA} \
    --model_path {LOCAL_OUTPUT} \
    --iteration {ITERATIONS}"""

print(f"\n📝 Komut:")
print(render_cmd)
print()

!{render_cmd}

# Render edilen videoyu bul
video_files = glob.glob(os.path.join(LOCAL_OUTPUT, "**/*.mp4"), recursive=True)

if video_files:
    print("\n" + "="*60)
    print("✅ Render tamamlandı!")
    print("="*60)
    print(f"\n📹 Video: {video_files[0]}")
    
    # Videoyu göster
    print("\n📺 Video oynatılıyor...\n")
    display(Video(video_files[0], width=800))
    
    # Drive'a kopyala
    scene_name = os.path.basename(LOCAL_DATA)
    drive_output = os.path.join(OUTPUT_BASE, scene_name)
    
    print(f"\n📤 Video Drive'a kopyalanıyor: {drive_output}")
    # Model zaten kopyalandı, sadece render klasörünü güncelle
    render_dir_local = os.path.dirname(video_files[0])
    render_dir_drive = os.path.join(drive_output, os.path.basename(render_dir_local))
    
    if os.path.exists(render_dir_drive):
        shutil.rmtree(render_dir_drive)
    shutil.copytree(render_dir_local, render_dir_drive)
    print(f"✅ Video Drive'a kopyalandı")
else:
    print("\n⚠️  Video dosyası bulunamadı. Çıktı klasörünü kontrol edin.")

print("\n📝 Sonraki adım: Cell 9 ile PLY export yapın (opsiyonel)")

## 💾 Cell 9: PLY Export (Opsiyonel)

Frame başına 3D Gaussian point cloud'ları export eder.

In [ ]:
EXPORT_PLY = False  # @param {type:"boolean"}

if EXPORT_PLY:
    print("="*60)
    print("💾 PLY Export")
    print("="*60)
    
    # Export komutu
    export_cmd = f"""python /content/4DGaussians-Enhanced/export_perframe_3DGS.py \
        --source_path {LOCAL_DATA} \
        --model_path {LOCAL_OUTPUT} \
        --iteration {ITERATIONS}"""
    
    print(f"\n📝 Komut:")
    print(export_cmd)
    print()
    
    !{export_cmd}
    
    ply_dir = os.path.join(LOCAL_OUTPUT, "per_frame_ply")
    if os.path.exists(ply_dir):
        ply_files = glob.glob(os.path.join(ply_dir, "*.ply"))
        
        print("\n" + "="*60)
        print("✅ Export tamamlandı!")
        print("="*60)
        print(f"\n💾 {len(ply_files)} PLY dosyası oluşturuldu")
        print(f"📁 Lokal konum: {ply_dir}")
        
        # Drive'a kopyala
        scene_name = os.path.basename(LOCAL_DATA)
        drive_output = os.path.join(OUTPUT_BASE, scene_name)
        ply_dir_drive = os.path.join(drive_output, "per_frame_ply")
        
        print(f"\n📤 PLY dosyaları Drive'a kopyalanıyor: {ply_dir_drive}")
        if os.path.exists(ply_dir_drive):
            shutil.rmtree(ply_dir_drive)
        shutil.copytree(ply_dir, ply_dir_drive)
        print(f"✅ PLY dosyaları Drive'a kopyalandı")
    else:
        print("\n⚠️  Export klasörü bulunamadı. Komut çıktısını kontrol edin.")
else:
    print("⏭️  PLY export atlandı (EXPORT_PLY=False)")
    print("   PLY export yapmak için EXPORT_PLY=True yapın ve tekrar çalıştırın")

print("\n" + "="*60)
print("🎉 Tüm işlemler tamamlandı!")
print("="*60)
print(f"\n📁 Çıktılar: {OUTPUT_BASE}")
print("\n✅ 4D Gaussian modeliniz hazır!")